# 🌟 Gold Layer Transformation Pipeline

This notebook transforms clean, validated data from the Silver layer into business-ready analytics tables in the Gold layer.

**What this pipeline does:**
- Reads trip data from Silver tables (Yellow, Green, FHV, FHVHV)
- Creates dimension tables (date, time, location, vendor, payment, rate, company)
- Builds the unified fact_trips table with proper dimension keys
- Generates analytical aggregate tables for fast querying
- Processes only new data incrementally using metadata tracking
- Follows star schema design principles



## 📦 Configuration & Setup

Let's define our data locations and pipeline settings.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, date
from functools import reduce

# Database configuration
DATABASE = "nyc_taxi_lakehouse"
PIPELINE_VERSION = "1.0"

# Silver layer tables (our source)
SILVER_TABLES = {
    "yellow": f"{DATABASE}.silver.yellow_trips",
    "green": f"{DATABASE}.silver.green_trips",
    "fhv": f"{DATABASE}.silver.fhv_trips",
    "fhvhv": f"{DATABASE}.silver.fhvhv_trips"
}

# Gold layer tables (our destination)
GOLD_SCHEMA = f"{DATABASE}.gold"

# Dimension tables
DIM_DATE = f"{GOLD_SCHEMA}.dim_date"
DIM_TIME = f"{GOLD_SCHEMA}.dim_time"
DIM_LOCATION = f"{GOLD_SCHEMA}.dim_location"
DIM_VENDOR = f"{GOLD_SCHEMA}.dim_vendor"
DIM_PAYMENT = f"{GOLD_SCHEMA}.dim_payment"
DIM_RATE = f"{GOLD_SCHEMA}.dim_rate"
DIM_COMPANY = f"{GOLD_SCHEMA}.dim_company"

# Fact table
FACT_TRIPS = f"{GOLD_SCHEMA}.fact_trips"

# Aggregate tables
AGG_BOROUGH = f"{GOLD_SCHEMA}.agg_borough_metrics"
AGG_ZONE = f"{GOLD_SCHEMA}.agg_zone_metrics"
AGG_VENDOR = f"{GOLD_SCHEMA}.agg_vendor_performance"
AGG_COMPANY = f"{GOLD_SCHEMA}.agg_company_performance"
AGG_REVENUE = f"{GOLD_SCHEMA}.agg_revenue_analysis"

# Metadata tracking
METADATA_TABLE = f"{GOLD_SCHEMA}.gold_processing_metadata"
ZONE_LOOKUP_TABLE = f"{DATABASE}.bronze.taxi_zone_lookup"

print("✅ Configuration loaded successfully!")
print(f"📊 Processing {len(SILVER_TABLES)} trip types")
print(f"🎯 Target schema: {GOLD_SCHEMA}")

## 🔍 Incremental Processing Logic

These functions help us figure out what's new and needs processing.

In [0]:
def get_processed_partitions():
    """
    Check which partitions we've already processed in the gold layer.
    Returns a set of (vehicle_type, year, month) tuples.
    """
    try:
        processed_df = spark.table(METADATA_TABLE).filter("status = 'SUCCESS'")
        
        # Extract partition info from the metadata
        # Extract vehicle type from fact_trips records
        processed = set()
        for row in processed_df.collect():
            if "fact_trips" in row.table_name:
                # Parse partition_key format: "yellow_2024-10" or "2024-10"
                parts = row.partition_key.split("-")
                if len(parts) >= 2:
                    # Try to extract vehicle type and year/month
                    if "_" in row.partition_key:
                        # Format: "yellow_2024-10"
                        vtype_year = row.partition_key.split("-")[0]
                        vtype = vtype_year.split("_")[0]
                        year = int(vtype_year.split("_")[1])
                        month = int(parts[1])
                    else:
                        # Format: "2024-10" - try to infer from other metadata
                        year = int(parts[0])
                        month = int(parts[1])
                        # Skip if we can't determine vehicle type
                        continue
                    processed.add((vtype, year, month))
        
        print(f"📋 Found {len(processed)} already processed partitions in gold layer")
        return processed
    except Exception as e:
        print(f"⚠️  No metadata found yet. Starting fresh!")
        return set()


def get_new_silver_partitions(vehicle_type, silver_table):
    """
    Find new partitions in the silver layer that haven't been processed yet.
    Uses the year/month columns that exist in silver tables.
    """
    # Get distinct year/month combinations from silver table
    silver_partitions = (
        spark.table(silver_table)
        .select("year", "month")
        .distinct()
        .collect()
    )
    
    # Convert to set of tuples
    available = {(vehicle_type, row.year, row.month) for row in silver_partitions}
    
    # Subtract already processed ones
    processed = get_processed_partitions()
    new_partitions = available - processed
    
    return sorted(list(new_partitions), key=lambda x: (x[1], x[2]))  # Sort by year, month


def record_processing_metadata(table_name, partition_key, row_count, source_count, 
                                 status="SUCCESS", error_msg=None, start_time=None):
    """
    Record metadata about what we just processed.
    """
    end_time = datetime.now()
    duration_sec = (end_time - start_time).total_seconds() if start_time else None
    
    metadata_record = spark.createDataFrame([(
        table_name,
        partition_key,
        end_time,
        row_count,
        source_count,
        None,  # data_quality_score - can be calculated later
        duration_sec,
        status,
        error_msg,
        PIPELINE_VERSION
    )], schema="""
        table_name STRING,
        partition_key STRING,
        processed_at TIMESTAMP,
        row_count LONG,
        source_row_count LONG,
        data_quality_score DOUBLE,
        processing_duration_sec LONG,
        status STRING,
        error_message STRING,
        pipeline_version STRING
    """)
    
    metadata_record.write.mode("append").saveAsTable(METADATA_TABLE)


print("✅ Incremental processing functions loaded!")

## 🏗️ Dimension Table Builders

These functions create our dimension tables. Dimensions are built once and can be reused.

In [0]:
def build_dim_date():
    """
    Create a comprehensive date dimension table.
    Generates dates from 2020 to 2030.
    """
    print("📅 Building date dimension...")
    
    # Generate date range
    from datetime import timedelta
    start_date = date(2020, 1, 1)
    end_date = date(2030, 12, 31)
    
    # Create list of dates
    date_list = []
    current_date = start_date
    while current_date <= end_date:
        date_list.append((current_date,))
        current_date += timedelta(days=1)
    
    # Create dataframe
    dates_df = spark.createDataFrame(date_list, ["date"])
    
    # Add all date attributes
    dim_date_df = dates_df.select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("date"),
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.weekofyear("date").alias("week_of_year"),
        F.dayofmonth("date").alias("day_of_month"),
        F.dayofweek("date").alias("day_of_week"),
        F.date_format("date", "EEEE").alias("day_name"),
        F.dayofweek("date").isin([1, 7]).alias("is_weekend"),
        F.lit(False).alias("is_holiday"),  # Placeholder - can be enriched later
        F.lit(None).cast("string").alias("holiday_name"),
        (~F.dayofweek("date").isin([1, 7])).alias("is_business_day"),
        F.year("date").alias("fiscal_year"),  # Assuming calendar year = fiscal year
        F.quarter("date").alias("fiscal_quarter")
    )
    
    # Write to gold layer
    dim_date_df.write.mode("overwrite").saveAsTable(DIM_DATE)
    count = dim_date_df.count()
    print(f"   ✅ Created {count:,} date records")
    
    return dim_date_df


def build_dim_time():
    """
    Create time of day dimension (hour and minute combinations).
    """
    print("⏰ Building time dimension...")
    
    # Generate all hour:minute combinations (1440 records for 24 hours * 60 minutes)
    time_list = [(h, m) for h in range(24) for m in range(60)]
    time_df = spark.createDataFrame(time_list, ["hour", "minute"])
    
    dim_time_df = time_df.select(
        (F.col("hour") * 100 + F.col("minute")).alias("time_key"),
        F.col("hour"),
        F.col("minute"),
        F.when((F.col("hour") >= 0) & (F.col("hour") <= 5), "Night")
         .when((F.col("hour") >= 6) & (F.col("hour") <= 11), "Morning")
         .when((F.col("hour") >= 12) & (F.col("hour") <= 17), "Afternoon")
         .when((F.col("hour") >= 18) & (F.col("hour") <= 23), "Evening")
         .alias("time_of_day"),
        F.when(F.col("hour") == 0, 12)
         .when(F.col("hour") > 12, F.col("hour") - 12)
         .otherwise(F.col("hour"))
         .alias("hour_12"),
        F.when(F.col("hour") < 12, "AM").otherwise("PM").alias("am_pm"),
        (F.col("hour").between(7, 9) | F.col("hour").between(17, 19)).alias("is_peak_hour"),
        (F.col("hour").between(9, 17)).alias("is_business_hours")
    )
    
    dim_time_df.write.mode("overwrite").saveAsTable(DIM_TIME)
    count = dim_time_df.count()
    print(f"   ✅ Created {count:,} time records")
    
    return dim_time_df


def build_dim_location():
    """
    Create location dimension from the zone lookup table.
    """
    print("🗺️  Building location dimension...")
    
    zone_df = spark.table(ZONE_LOOKUP_TABLE)
    
    dim_location_df = zone_df.select(
        F.col("LocationID").alias("location_key"),
        F.col("LocationID").alias("location_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone"),
        F.col("service_zone"),
        F.col("Zone").rlike("Airport").alias("is_airport"),
        (F.col("Borough") == "Manhattan").alias("is_manhattan"),
        F.when(F.col("Zone").rlike("Airport"), "Airport")
         .when(F.col("service_zone") == "Boro Zone", "Residential")
         .otherwise("Downtown")
         .alias("zone_type")
    )
    
    dim_location_df.write.mode("overwrite").saveAsTable(DIM_LOCATION)
    count = dim_location_df.count()
    print(f"   ✅ Created {count:,} location records")
    
    return dim_location_df


def build_dim_vendor():
    """
    Create vendor dimension from known vendor IDs.
    """
    print("🚖 Building vendor dimension...")
    
    vendors = [
        (1, "Creative Mobile Technologies, LLC", 1, ["yellow", "green"]),
        (2, "Curb Mobility, LLC", 2, ["yellow", "green"]),
        (6, "Myle Technologies Inc", 6, ["yellow"])
    ]
    
    # Create with explicit schema to ensure IntegerType for vendor_key
    schema = StructType([
        StructField("vendor_key", IntegerType(), False),
        StructField("vendor_name", StringType(), False),
        StructField("vendor_code", IntegerType(), False),
        StructField("vehicle_types_served", ArrayType(StringType()), False)
    ])
    
    dim_vendor_df = spark.createDataFrame(vendors, schema)
    
    dim_vendor_df.write.mode("overwrite").saveAsTable(DIM_VENDOR)
    count = dim_vendor_df.count()
    print(f"   ✅ Created {count:,} vendor records")
    
    return dim_vendor_df


def build_dim_payment():
    """
    Create payment type dimension.
    """
    print("💳 Building payment dimension...")
    
    payments = [
        (0, "Flex Fare", 0, False),
        (1, "Credit Card", 1, True),
        (2, "Cash", 2, False),
        (3, "No Charge", 3, False),
        (4, "Dispute", 4, False),
        (5, "Unknown", 5, False)
    ]
    
    # Create with explicit schema to ensure IntegerType for payment_key
    schema = StructType([
        StructField("payment_key", IntegerType(), False),
        StructField("payment_method", StringType(), False),
        StructField("payment_code", IntegerType(), False),
        StructField("allows_tipping", BooleanType(), False)
    ])
    
    dim_payment_df = spark.createDataFrame(payments, schema)
    
    dim_payment_df.write.mode("overwrite").saveAsTable(DIM_PAYMENT)
    count = dim_payment_df.count()
    print(f"   ✅ Created {count:,} payment records")
    
    return dim_payment_df


def build_dim_rate():
    """
    Create rate code dimension.
    """
    print("💰 Building rate dimension...")
    
    rates = [
        (1, "Standard Rate", 1, False),
        (2, "JFK", 2, True),
        (3, "Newark", 3, True),
        (4, "Nassau or Westchester", 4, False),
        (5, "Negotiated Fare", 5, False),
        (6, "Group Ride", 6, False),
        (99, "Null/Unknown", 99, False)
    ]
    
    # Create with explicit schema to ensure IntegerType for rate_key
    schema = StructType([
        StructField("rate_key", IntegerType(), False),
        StructField("rate_type", StringType(), False),
        StructField("rate_code", IntegerType(), False),
        StructField("is_airport_rate", BooleanType(), False)
    ])
    
    dim_rate_df = spark.createDataFrame(rates, schema)
    
    dim_rate_df.write.mode("overwrite").saveAsTable(DIM_RATE)
    count = dim_rate_df.count()
    print(f"   ✅ Created {count:,} rate records")
    
    return dim_rate_df


def build_dim_company():
    """
    Create company dimension for FHVHV services.
    """
    print("🚗 Building company dimension...")
    
    companies = [
        (1, "Uber", "HV0003", "FHVHV"),
        (2, "Lyft", "HV0005", "FHVHV"),
        (3, "Unknown", "Unknown", "FHVHV")
    ]
    
    # Create with explicit schema to ensure IntegerType for company_key
    schema = StructType([
        StructField("company_key", IntegerType(), False),
        StructField("company_name", StringType(), False),
        StructField("license_number", StringType(), False),
        StructField("service_type", StringType(), False)
    ])
    
    dim_company_df = spark.createDataFrame(companies, schema)
    
    dim_company_df.write.mode("overwrite").saveAsTable(DIM_COMPANY)
    count = dim_company_df.count()
    print(f"   ✅ Created {count:,} company records")
    
    return dim_company_df


print("✅ Dimension table builders loaded!")

## 🎯 Fact Table Builder

This is the heart of our gold layer - the unified trip fact table with all vehicle types.

In [0]:
def transform_to_fact_trips(silver_df, vehicle_type, dim_vendor_df, dim_payment_df, 
                             dim_rate_df, dim_company_df):
    """
    Transform silver trips into gold fact table format.
    Handles differences between trip types gracefully.
    """
    
    # Generate unique trip_id
    silver_df = silver_df.withColumn(
        "trip_id",
        F.concat(
            F.lit(vehicle_type), F.lit("_"),
            F.col("year").cast("string"), F.lit("_"),
            F.col("month").cast("string"), F.lit("_"),
            F.monotonically_increasing_id().cast("string")
        )
    )
    
    # Add vehicle type
    silver_df = silver_df.withColumn("vehicle_type", F.lit(vehicle_type))
    
    # Create dimension keys
    silver_df = silver_df.withColumn(
        "date_key",
        F.date_format("pickup_datetime", "yyyyMMdd").cast("int")
    )
    
    silver_df = silver_df.withColumn(
        "time_key",
        (F.hour("pickup_datetime") * 100 + F.minute("pickup_datetime")).cast("int")
    )
    
    # Cast location IDs to INT to ensure schema consistency across all trip types
    silver_df = silver_df.withColumn(
        "pickup_location_key",
        F.col("pickup_location_id").cast("int")
    )
    
    silver_df = silver_df.withColumn(
        "dropoff_location_key",
        F.col("dropoff_location_id").cast("int")
    )
    
    # Vendor key (only for yellow and green)
    # IMPORTANT: Cast to int to ensure schema consistency across all trip types
    # This prevents Delta Lake schema merge errors (LongType vs IntegerType conflicts)
    if vehicle_type in ["yellow", "green"]:
        silver_df = silver_df.join(
            dim_vendor_df.select("vendor_key", F.col("vendor_code").alias("vendor_id")),
            "vendor_id",
            "left"
        )
        # Explicitly cast vendor_key to int after join to ensure type consistency
        silver_df = silver_df.withColumn("vendor_key", F.col("vendor_key").cast("int"))
    else:
        silver_df = silver_df.withColumn("vendor_key", F.lit(None).cast("int"))
    
    # Payment key (only for yellow and green)
    if vehicle_type in ["yellow", "green"]:
        silver_df = silver_df.join(
            dim_payment_df.select("payment_key", F.col("payment_code").alias("payment_type_id")),
            "payment_type_id",
            "left"
        )
        # Explicitly cast payment_key to int after join to ensure type consistency
        silver_df = silver_df.withColumn("payment_key", F.col("payment_key").cast("int"))
    else:
        silver_df = silver_df.withColumn("payment_key", F.lit(None).cast("int"))
    
    # Rate key (only for yellow and green)
    if vehicle_type in ["yellow", "green"]:
        silver_df = silver_df.join(
            dim_rate_df.select("rate_key", F.col("rate_code").alias("rate_code_id")),
            "rate_code_id",
            "left"
        )
        # Explicitly cast rate_key to int after join to ensure type consistency
        silver_df = silver_df.withColumn("rate_key", F.col("rate_key").cast("int"))
    else:
        silver_df = silver_df.withColumn("rate_key", F.lit(None).cast("int"))
    
    # Company key (only for FHVHV)
    if vehicle_type == "fhvhv":
        # Create a temp column for the join
        company_lookup = dim_company_df.select(
            F.col("company_key"),
            F.col("company_name").alias("comp_name")
        )
        silver_df = silver_df.join(
            company_lookup,
            silver_df.company_name == company_lookup.comp_name,
            "left"
        ).drop("comp_name")
        # Explicitly cast company_key to int after join to ensure type consistency
        silver_df = silver_df.withColumn("company_key", F.col("company_key").cast("int"))
    else:
        silver_df = silver_df.withColumn("company_key", F.lit(None).cast("int"))
    
    # CRITICAL: Cast all numeric columns to consistent types to avoid Delta schema conflicts
    # Different trip types may have different source types, so we standardize here
    
    # Cast location IDs to INT (already done above, but keeping for clarity)
    # Cast all financial/metric columns to DOUBLE for consistency
    numeric_double_cols = [
        "trip_distance_miles", "trip_duration_seconds", "trip_duration_minutes", 
        "average_speed_mph", "fare_amount", "extra_charges", "mta_tax", 
        "tip_amount", "tolls_amount", "improvement_surcharge", 
        "congestion_surcharge", "airport_fee", "total_fare", "tip_percentage",
        "base_passenger_fare", "black_car_fund", "sales_tax", "driver_pay"
    ]
    
    for col_name in numeric_double_cols:
        if col_name in silver_df.columns:
            silver_df = silver_df.withColumn(col_name, F.col(col_name).cast("double"))
    
    # Cast passenger_count and other integer columns
    integer_cols = ["passenger_count", "vendor_id", "payment_type_id", "rate_code_id"]
    
    for col_name in integer_cols:
        if col_name in silver_df.columns:
            silver_df = silver_df.withColumn(col_name, F.col(col_name).cast("int"))
    
    # Now select the fact table columns in the right order
    # Handle columns that might not exist for all trip types
    
    fact_df = silver_df.select(
        # Trip identification
        F.col("trip_id"),
        F.col("vehicle_type"),
        
        # Temporal information
        F.col("pickup_datetime"),
        F.col("dropoff_datetime"),
        F.col("date_key"),
        F.col("time_key"),
        F.col("year"),
        F.col("month"),
        
        # Location information
        F.col("pickup_location_key"),
        F.col("dropoff_location_key"),
        
        # Trip metrics
        F.col("trip_distance_miles") if "trip_distance_miles" in silver_df.columns else F.lit(None).cast("double").alias("trip_distance_miles"),
        F.col("trip_duration_seconds"),
        F.col("trip_duration_minutes"),
        F.col("average_speed_mph") if "average_speed_mph" in silver_df.columns else F.lit(None).cast("double").alias("average_speed_mph"),
        
        # Passenger information
        F.col("passenger_count") if "passenger_count" in silver_df.columns else F.lit(None).cast("int").alias("passenger_count"),
        
        # Financial metrics
        F.col("fare_amount") if "fare_amount" in silver_df.columns else F.lit(None).cast("double").alias("fare_amount"),
        F.col("extra_charges") if "extra_charges" in silver_df.columns else F.lit(None).cast("double").alias("extra_charges"),
        F.col("mta_tax") if "mta_tax" in silver_df.columns else F.lit(None).cast("double").alias("mta_tax"),
        F.col("tip_amount") if "tip_amount" in silver_df.columns else F.lit(None).cast("double").alias("tip_amount"),
        F.col("tolls_amount") if "tolls_amount" in silver_df.columns else F.lit(None).cast("double").alias("tolls_amount"),
        F.col("improvement_surcharge") if "improvement_surcharge" in silver_df.columns else F.lit(None).cast("double").alias("improvement_surcharge"),
        F.col("congestion_surcharge") if "congestion_surcharge" in silver_df.columns else F.lit(None).cast("double").alias("congestion_surcharge"),
        F.col("airport_fee") if "airport_fee" in silver_df.columns else F.lit(None).cast("double").alias("airport_fee"),
        F.col("total_fare") if "total_fare" in silver_df.columns else F.lit(None).cast("double").alias("total_fare"),
        F.col("tip_percentage") if "tip_percentage" in silver_df.columns else F.lit(None).cast("double").alias("tip_percentage"),
        
        # Vendor/Company/Payment/Rate keys
        F.col("vendor_key"),
        F.col("company_key"),
        F.col("payment_key"),
        F.col("rate_key"),
        
        # Service-specific fields
        F.col("dispatch_type") if "dispatch_type" in silver_df.columns else F.lit(None).cast("string").alias("dispatch_type"),
        F.col("shared_match_flag").alias("shared_trip") if "shared_match_flag" in silver_df.columns else F.lit(None).cast("boolean").alias("shared_trip"),
        F.col("wav_request_flag").alias("wav_request") if "wav_request_flag" in silver_df.columns else F.lit(None).cast("boolean").alias("wav_request"),
        F.col("wav_match_flag").alias("wav_match") if "wav_match_flag" in silver_df.columns else F.lit(None).cast("boolean").alias("wav_match"),
        
        # Data quality
        F.col("data_quality_flags"),
        F.col("is_suspicious"),
        
        # Metadata
        F.col("processed_at")
    )
    
    return fact_df


print("✅ Fact table transformation function loaded!")

## 📊 Aggregate Table Builders

These functions create pre-computed analytics tables for fast business queries.

In [0]:
def build_agg_borough_metrics(fact_df, dim_location_df):
    """
    Create borough-level metrics aggregate.
    """
    print("🏙️  Building borough metrics aggregate...")
    
    # Join with location for borough info
    df_with_location = fact_df.join(
        dim_location_df.select(
            F.col("location_key").alias("pickup_loc_key"),
            F.col("borough").alias("pickup_borough")
        ),
        fact_df.pickup_location_key == F.col("pickup_loc_key"),
        "left"
    ).join(
        dim_location_df.select(
            F.col("location_key").alias("dropoff_loc_key"),
            F.col("borough").alias("dropoff_borough")
        ),
        fact_df.dropoff_location_key == F.col("dropoff_loc_key"),
        "left"
    )
    
    # Aggregate by borough
    agg_df = df_with_location.groupBy(
        "pickup_borough", "year", "month", "vehicle_type"
    ).agg(
        F.count("*").alias("total_pickups"),
        F.sum(F.when(F.col("total_fare").isNotNull(), 1).otherwise(0)).alias("total_trips"),
        F.sum("total_fare").alias("total_revenue"),
        F.avg("total_fare").alias("avg_fare_per_trip"),
        F.sum("tip_amount").alias("total_tips"),
        F.avg("tip_percentage").alias("avg_tip_percentage"),
        F.avg("trip_distance_miles").alias("avg_trip_distance"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration"),
        F.avg("average_speed_mph").alias("avg_speed_mph")
    ).withColumnRenamed("pickup_borough", "borough")
    
    # Count dropoffs per borough
    dropoff_counts = df_with_location.groupBy(
        "dropoff_borough", "year", "month", "vehicle_type"
    ).agg(
        F.count("*").alias("total_dropoffs")
    ).withColumnRenamed("dropoff_borough", "borough")
    
    # Join pickup and dropoff metrics
    agg_df = agg_df.join(
        dropoff_counts,
        ["borough", "year", "month", "vehicle_type"],
        "left"
    ).fillna(0, ["total_dropoffs"])
    
    # Add rankings
    window_revenue = Window.partitionBy("year", "month", "vehicle_type").orderBy(F.desc("total_revenue"))
    window_volume = Window.partitionBy("year", "month", "vehicle_type").orderBy(F.desc("total_trips"))
    
    agg_df = agg_df.withColumn("revenue_rank", F.row_number().over(window_revenue))
    agg_df = agg_df.withColumn("volume_rank", F.row_number().over(window_volume))
    
    # Add growth metrics (placeholders - would need historical data)
    agg_df = agg_df.withColumn("mom_trip_growth_pct", F.lit(None).cast("double"))
    agg_df = agg_df.withColumn("yoy_trip_growth_pct", F.lit(None).cast("double"))
    
    return agg_df


def build_agg_zone_metrics(fact_df, dim_location_df):
    """
    Create zone-level metrics aggregate.
    """
    print("📍 Building zone metrics aggregate...")
    
    # Join with location
    df_with_location = fact_df.join(
        dim_location_df.select(
            F.col("location_key").alias("pickup_loc_key"),
            F.col("zone").alias("pickup_zone"),
            F.col("borough").alias("pickup_borough")
        ),
        fact_df.pickup_location_key == F.col("pickup_loc_key"),
        "left"
    ).join(
        dim_location_df.select(
            F.col("location_key").alias("dropoff_loc_key"),
            F.col("zone").alias("dropoff_zone")
        ),
        fact_df.dropoff_location_key == F.col("dropoff_loc_key"),
        "left"
    )
    
    # Get pickup metrics by zone
    pickup_metrics = df_with_location.groupBy(
        "pickup_loc_key", "pickup_zone", "pickup_borough", "year", "month"
    ).agg(
        F.count("*").alias("total_pickups"),
        F.sum("total_fare").alias("total_revenue"),
        F.avg("total_fare").alias("avg_fare")
    ).withColumnRenamed("pickup_loc_key", "location_key") \
     .withColumnRenamed("pickup_zone", "zone") \
     .withColumnRenamed("pickup_borough", "borough")
    
    # Get dropoff counts by zone
    dropoff_counts = df_with_location.groupBy(
        "dropoff_loc_key", "year", "month"
    ).agg(
        F.count("*").alias("total_dropoffs")
    ).withColumnRenamed("dropoff_loc_key", "location_key")
    
    # Peak hour analysis
    peak_hours = df_with_location.withColumn(
        "hour", F.hour("pickup_datetime")
    ).groupBy(
        "pickup_loc_key", "year", "month", "hour"
    ).agg(
        F.count("*").alias("hour_trips")
    )
    
    window_peak = Window.partitionBy("pickup_loc_key", "year", "month").orderBy(F.desc("hour_trips"))
    peak_hour_info = peak_hours.withColumn(
        "rank", F.row_number().over(window_peak)
    ).filter("rank = 1").select(
        F.col("pickup_loc_key").alias("location_key"),
        "year", "month",
        F.col("hour").alias("peak_hour"),
        F.col("hour_trips").alias("peak_hour_trips")
    )
    
    # Top destinations
    top_destinations = df_with_location.groupBy(
        "pickup_loc_key", "year", "month", "dropoff_zone"
    ).agg(
        F.count("*").alias("dest_trips")
    )
    
    window_dest = Window.partitionBy("pickup_loc_key", "year", "month").orderBy(F.desc("dest_trips"))
    top_dest_info = top_destinations.withColumn(
        "rank", F.row_number().over(window_dest)
    ).filter("rank = 1").select(
        F.col("pickup_loc_key").alias("location_key"),
        "year", "month",
        F.col("dropoff_zone").alias("top_destination_zone"),
        F.col("dest_trips").alias("top_destination_trip_count")
    )
    
    # Combine all metrics
    agg_df = pickup_metrics.join(
        dropoff_counts, ["location_key", "year", "month"], "left"
    ).join(
        peak_hour_info, ["location_key", "year", "month"], "left"
    ).join(
        top_dest_info, ["location_key", "year", "month"], "left"
    ).fillna(0, ["total_dropoffs"])
    
    return agg_df


def build_agg_vendor_performance(fact_df, dim_vendor_df):
    """
    Create vendor performance metrics (Yellow/Green only).
    """
    print("🚕 Building vendor performance aggregate...")
    
    # Filter for yellow and green only
    vendor_trips = fact_df.filter(
        F.col("vehicle_type").isin(["yellow", "green"]) & 
        F.col("vendor_key").isNotNull()
    )
    
    # Join with vendor dimension
    df_with_vendor = vendor_trips.join(
        dim_vendor_df.select("vendor_key", "vendor_name"),
        "vendor_key",
        "left"
    )
    
    # Calculate metrics
    agg_df = df_with_vendor.groupBy(
        "vendor_name", "vehicle_type", "year", "month"
    ).agg(
        F.count("*").alias("total_trips"),
        F.sum("total_fare").alias("total_revenue"),
        F.avg("total_fare").alias("avg_fare"),
        F.avg("tip_percentage").alias("avg_tip_percentage"),
        F.avg("trip_distance_miles").alias("avg_trip_distance"),
        F.avg(F.when(~F.col("is_suspicious"), 1.0).otherwise(0.0)).alias("data_quality_score")
    )
    
    # Calculate market share
    window_market = Window.partitionBy("vehicle_type", "year", "month")
    agg_df = agg_df.withColumn(
        "market_share_pct",
        (F.col("total_trips") / F.sum("total_trips").over(window_market)) * 100
    )
    
    return agg_df


def build_agg_company_performance(fact_df, dim_company_df):
    """
    Create company performance metrics (FHVHV - Uber/Lyft).
    """
    print("🚙 Building company performance aggregate...")
    
    # Filter for FHVHV only
    company_trips = fact_df.filter(
        (F.col("vehicle_type") == "fhvhv") & 
        F.col("company_key").isNotNull()
    )
    
    # Join with company dimension
    df_with_company = company_trips.join(
        dim_company_df.select("company_key", "company_name"),
        "company_key",
        "left"
    )
    
    # Calculate metrics
    agg_df = df_with_company.groupBy(
        "company_name", "year", "month"
    ).agg(
        F.count("*").alias("total_trips"),
        F.sum("total_fare").alias("total_revenue"),  # Using total_fare as base_passenger_fare equivalent
        F.avg("total_fare").alias("avg_base_fare"),
        F.avg("tip_amount").alias("avg_tip"),
        (F.avg(F.when(F.col("shared_trip") == True, 1.0).otherwise(0.0)) * 100).alias("shared_trip_pct"),
        (F.avg(F.when(F.col("wav_request") == True, 1.0).otherwise(0.0)) * 100).alias("wav_request_pct"),
        F.countDistinct("pickup_location_key").alias("unique_zones_served")
    )
    
    # Calculate market share
    window_market = Window.partitionBy("year", "month")
    agg_df = agg_df.withColumn(
        "market_share_pct",
        (F.col("total_trips") / F.sum("total_trips").over(window_market)) * 100
    )
    
    # Add placeholders for fields we can't calculate
    agg_df = agg_df.withColumn("avg_driver_pay", F.lit(None).cast("double"))
    agg_df = agg_df.withColumn("avg_wait_time_seconds", F.lit(None).cast("double"))
    agg_df = agg_df.withColumn("manhattan_trip_pct", F.lit(None).cast("double"))
    
    return agg_df


def build_agg_revenue_analysis(fact_df):
    """
    Create revenue analysis aggregate.
    """
    print("💵 Building revenue analysis aggregate...")
    
    # Filter for records with revenue data
    revenue_df = fact_df.filter(F.col("total_fare").isNotNull())
    
    agg_df = revenue_df.groupBy(
        "year", "month", "vehicle_type"
    ).agg(
        F.sum("total_fare").alias("total_revenue"),
        F.sum("fare_amount").alias("base_fare_revenue"),
        F.sum("tip_amount").alias("tip_revenue"),
        F.sum(
            F.coalesce("congestion_surcharge", F.lit(0)) + 
            F.coalesce("airport_fee", F.lit(0)) + 
            F.coalesce("improvement_surcharge", F.lit(0))
        ).alias("surcharge_revenue"),
        F.sum("tolls_amount").alias("tolls_revenue"),
        F.sum("mta_tax").alias("mta_tax_revenue"),
        F.avg("total_fare").alias("revenue_per_trip"),
        F.avg(
            F.when(F.col("trip_distance_miles") > 0, 
                   F.col("total_fare") / F.col("trip_distance_miles"))
        ).alias("revenue_per_mile"),
        F.avg(
            F.when(F.col("trip_duration_minutes") > 0,
                   F.col("total_fare") / F.col("trip_duration_minutes"))
        ).alias("revenue_per_minute")
    )
    
    # Payment mix (only for vehicle types with payment data)
    payment_mix = fact_df.filter(
        F.col("payment_key").isNotNull() & 
        F.col("vehicle_type").isin(["yellow", "green"])
    ).groupBy(
        "year", "month", "vehicle_type"
    ).agg(
        (F.avg(
            F.when(F.col("payment_key") == 1, 1.0).otherwise(0.0)
        ) * 100).alias("credit_card_pct"),
        (F.avg(
            F.when(F.col("payment_key") == 2, 1.0).otherwise(0.0)
        ) * 100).alias("cash_pct")
    )
    
    # Join payment mix
    agg_df = agg_df.join(
        payment_mix,
        ["year", "month", "vehicle_type"],
        "left"
    )
    
    # Growth metrics (placeholders)
    agg_df = agg_df.withColumn("mom_revenue_growth_pct", F.lit(None).cast("double"))
    agg_df = agg_df.withColumn("yoy_revenue_growth_pct", F.lit(None).cast("double"))
    
    return agg_df


print("✅ Aggregate table builders loaded!")

## 🚀 Main Pipeline Execution

Here's where everything comes together!

### Step 1: Build All Dimension Tables

Dimensions are the foundation - we build them first and only once.

In [0]:
print("\n" + "="*80)
print("🏗️  BUILDING DIMENSION TABLES")
print("="*80 + "\n")

# Build all dimensions
dim_date_df = build_dim_date()
dim_time_df = build_dim_time()
dim_location_df = build_dim_location()
dim_vendor_df = build_dim_vendor()
dim_payment_df = build_dim_payment()
dim_rate_df = build_dim_rate()
dim_company_df = build_dim_company()

print("\n✅ All dimension tables created successfully!")

### Step 2: Process New Data into Fact Table

Only process partitions that haven't been loaded to gold yet.

In [0]:
print("\n" + "="*80)
print("🎯 LOADING FACT TABLE (INCREMENTAL)")
print("="*80 + "\n")

# Track what we process
total_trips_processed = 0

# Process each vehicle type
for vehicle_type, silver_table in SILVER_TABLES.items():
    
    print(f"\n🚗 Processing {vehicle_type.upper()} trips...")
    
    # Find new partitions to process
    new_partitions = get_new_silver_partitions(vehicle_type, silver_table)
    
    if not new_partitions:
        print(f"   ℹ️  No new data to process for {vehicle_type}")
        continue
    
    print(f"   📦 Found {len(new_partitions)} new partition(s) to process")
    
    # Process each partition
    for vtype, year, month in new_partitions:
        try:
            start_time = datetime.now()
            partition_key = f"{vehicle_type}_{year}-{month:02d}"
            
            print(f"\n   📅 Processing {vehicle_type} {partition_key}...")
            
            # Read silver data for this partition
            silver_df = spark.table(silver_table).filter(
                (F.col("year") == year) & (F.col("month") == month)
            )
            
            source_count = silver_df.count()
            print(f"      📊 Silver records: {source_count:,}")
            
            # Transform to fact table format
            fact_df = transform_to_fact_trips(
                silver_df, vehicle_type,
                dim_vendor_df, dim_payment_df, dim_rate_df, dim_company_df
            )
            
            row_count = fact_df.count()
            print(f"      ✨ Gold records: {row_count:,}")
            
            # Write to fact table (append mode)
            fact_df.write \
                .mode("append") \
                .partitionBy("year", "month", "vehicle_type") \
                .saveAsTable(FACT_TRIPS)
            
            print(f"      💾 Written to {FACT_TRIPS}")
            
            # Record metadata
            record_processing_metadata(
                FACT_TRIPS, partition_key, row_count, source_count,
                status="SUCCESS", start_time=start_time
            )
            
            total_trips_processed += row_count
            print(f"      ✅ Successfully processed {vehicle_type} {partition_key}")
            
        except Exception as e:
            print(f"      ❌ ERROR processing {vehicle_type} {partition_key}: {str(e)}")
            record_processing_metadata(
                FACT_TRIPS, partition_key, 0, 0,
                status="FAILED", error_msg=str(e), start_time=start_time
            )

print(f"\n\n✅ Fact table loading complete! Processed {total_trips_processed:,} total trips")

### Step 3: Build Aggregate Tables

Create pre-computed analytics tables from the fact table for fast queries.

In [0]:
print("\n" + "="*80)
print("📊 BUILDING AGGREGATE TABLES")
print("="*80 + "\n")

# Read the complete fact table
print("📖 Reading fact table...")
fact_df = spark.table(FACT_TRIPS)
fact_count = fact_df.count()
print(f"   Found {fact_count:,} total trips in fact table\n")

# Build each aggregate
try:
    # Borough metrics
    agg_borough_df = build_agg_borough_metrics(fact_df, dim_location_df)
    agg_borough_df.write.mode("overwrite") \
        .partitionBy("year", "month") \
        .saveAsTable(AGG_BOROUGH)
    print(f"   ✅ Saved to {AGG_BOROUGH}\n")
    
    # Zone metrics
    agg_zone_df = build_agg_zone_metrics(fact_df, dim_location_df)
    agg_zone_df.write.mode("overwrite") \
        .partitionBy("year", "month") \
        .saveAsTable(AGG_ZONE)
    print(f"   ✅ Saved to {AGG_ZONE}\n")
    
    # Vendor performance
    agg_vendor_df = build_agg_vendor_performance(fact_df, dim_vendor_df)
    agg_vendor_df.write.mode("overwrite") \
        .partitionBy("year", "month") \
        .saveAsTable(AGG_VENDOR)
    print(f"   ✅ Saved to {AGG_VENDOR}\n")
    
    # Company performance
    agg_company_df = build_agg_company_performance(fact_df, dim_company_df)
    agg_company_df.write.mode("overwrite") \
        .partitionBy("year", "month") \
        .saveAsTable(AGG_COMPANY)
    print(f"   ✅ Saved to {AGG_COMPANY}\n")
    
    # Revenue analysis
    agg_revenue_df = build_agg_revenue_analysis(fact_df)
    agg_revenue_df.write.mode("overwrite") \
        .partitionBy("year", "month") \
        .saveAsTable(AGG_REVENUE)
    print(f"   ✅ Saved to {AGG_REVENUE}\n")
    
    print("✅ All aggregate tables created successfully!")
    
except Exception as e:
    print(f"❌ Error building aggregates: {str(e)}")

## 🎉 Pipeline Complete!

Let's see what we built.

In [0]:
print("\n" + "="*80)
print("🎊 GOLD LAYER PIPELINE COMPLETE")
print("="*80 + "\n")

print("📋 Summary of created tables:\n")

print("🔷 Dimension Tables:")
for table in [DIM_DATE, DIM_TIME, DIM_LOCATION, DIM_VENDOR, DIM_PAYMENT, DIM_RATE, DIM_COMPANY]:
    count = spark.table(table).count()
    print(f"   • {table}: {count:,} records")

print("\n🔶 Fact Table:")
fact_count = spark.table(FACT_TRIPS).count()
print(f"   • {FACT_TRIPS}: {fact_count:,} trips")

print("\n🔸 Aggregate Tables:")
for table in [AGG_BOROUGH, AGG_ZONE, AGG_VENDOR, AGG_COMPANY, AGG_REVENUE]:
    try:
        count = spark.table(table).count()
        print(f"   • {table}: {count:,} records")
    except:
        print(f"   • {table}: Not created")

print("\n" + "="*80)
print("✨ Your gold layer is ready for analytics!")
print("="*80)

## 📝 Next Steps

**To process new data in the future:**
1. Just run this notebook again!
2. The pipeline will automatically detect new partitions in silver
3. Only new data will be processed and added to gold
4. Aggregates will be rebuilt to include the new data

**The pipeline handles:**
- ✅ Incremental processing (only new data)
- ✅ Schema differences between trip types
- ✅ Dimension key lookups
- ✅ Metadata tracking
- ✅ Error handling

**For monitoring:**
- Check `gold_processing_metadata` table for processing history
- Look for `status = 'FAILED'` to catch issues
- Review row counts to ensure data quality

**For optimization:**
- Run `OPTIMIZE` on fact_trips regularly
- Consider adding Z-ORDERING: `OPTIMIZE fact_trips ZORDER BY (date_key, pickup_location_key)`
- Monitor query performance and add indexes as needed

---

**Happy analyzing! 🎯**